In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Ensure Python can find src/
notebook_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
src_dir = os.path.abspath(os.path.join(notebook_dir, "..", "src"))
if src_dir not in sys.path:
    sys.path.append(src_dir)

from microgrid import MicroGrid

# Initialize the micro-grid (this will print your determinant and condition number)
mg = MicroGrid()

# Test a sample day (e.g., D1 = 30 kWh daytime load, D2 = 25 kWh critical load)
solar_x, battery_y = mg.solve_day(d1=30.0, d2=25.0)
print(f"\nTest Dispatch Result -> Solar (x): {solar_x:.2f} kWh | Battery (y): {battery_y:.2f} kWh")

--- Micro-Grid System Diagnostics ---
Determinant: -5.0000
Condition Number (np.linalg.cond): 5.8284

Test Dispatch Result -> Solar (x): 4.00 kWh | Battery (y): 9.00 kWh


In [ ]:
import pandas as pd
from microgrid import MicroGrid

mg = MicroGrid()

def get_validated_load(prompt_text: str) -> float:
    """Robust interactive validation for single-day input."""
    while True:
        raw = input(prompt_text).strip()
        if not raw:
            print("❌ Input cannot be empty. Please enter a valid number.")
            continue
        try:
            value = float(raw)
            if value < 0:
                print("❌ Load values cannot be negative. Please try again.")
                continue
            return value
        except ValueError:
            print("❌ Invalid input. Please enter a numeric value.")

# --- CHOOSE YOUR INPUT MODE ---
mode = input("Select input mode - Type 'interactive' or 'csv': ").strip().lower()

if mode == 'interactive':
    print("\n--- Interactive Single-Day Dispatch ---")
    d1 = get_validated_load("Enter Daytime Load D1 (kWh): ")
    d2 = get_validated_load("Enter Critical Load D2 (kWh): ")
    
    solar, battery = mg.solve_day(d1, d2)
    print(f"\nResults -> Solar (x): {solar:.2f} kWh | Battery (y): {battery:.2f} kWh")

elif mode == 'csv':
    print("\n--- Batch 30-Day CSV Processing ---")
    df = pd.read_csv("../data/microgrid_demand_30days.csv")
    
    # Process all days using your MicroGrid class
    results = [mg.solve_day(row['Daytime_Load_D1'], row['Critical_Load_D2']) for _, row in df.iterrows()]
    df['Solar_x'] = [r[0] for r in results]
    df['Battery_y'] = [r[1] for r in results]
    
    display(df.head())
else:
    print("❌ Unknown mode selected.")

--- Micro-Grid System Diagnostics ---
Determinant: -5.0000
Condition Number (np.linalg.cond): 5.8284
